<a href="https://colab.research.google.com/github/wuhao007/haowu999/blob/main/stocks_haowu999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 全球明星股 AHR999 偏离度分析
本工具将 AHR999 逻辑应用于你关注的股票清单，通过对数回归评估价格相对于长期增长趋势和 MA200 的偏离程度。

In [ ]:
!pip install yfinance --quiet
import datetime
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LinearRegression
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class Stock999:
    def __init__(self, ticker, name, start_date='2010-01-01'):
        self.ticker = ticker
        self.name = name
        self.start_date = pd.to_datetime(start_date)
        self.prices = None
        self.w = None
        self.b = None

    def load_data(self):
        # 统一处理代码格式
        try:
            df = yf.download(self.ticker, start=self.start_date, progress=False)
            if df.empty: return None
            df = df.reset_index()
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df[['Date', 'Close']].copy()
            df.columns = ['Date', 'Close']
            self.prices = df.dropna()
            self._fit_model()
            return self.prices
        except Exception as e:
            print(f"Error loading {self.ticker}: {e}")
            return None

    def _fit_model(self):
        df = self.prices.copy()
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df = df[(df['Days'] > 0) & (df['Close'] > 0.01)].copy()
        if len(df) < 50: return
        
        x = np.log10(df['Days'].values).reshape(-1, 1)
        y = np.log10(df['Close'].values)
        model = LinearRegression().fit(x, y)
        self.w = model.coef_[0]
        self.b = model.intercept_
        self.score = model.score(x, y)

    def calculate(self):
        if self.prices is None or self.w is None: return None
        df = self.prices.copy()
        df['MA200'] = df['Close'].rolling(200).mean()
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df['FitPrice'] = 10 ** (self.w * np.log10(df['Days']) + self.b)
        df['AHR999'] = (df['Close'] / df['MA200']) * (df['Close'] / df['FitPrice'])
        return df.dropna()

In [ ]:
stock_list = [
    ('0700.HK', '腾讯控股'),
    ('600519.SS', '贵州茅台'),
    ('AAPL', 'Apple'),
    ('ASML', 'ASML'),
    ('BABA', 'Alibaba ADR'),
    ('BRK-B', 'Berkshire B'),
    ('NTDOY', 'Nintendo ADR'),
    ('NVDA', 'NVIDIA'),
    ('OXY', 'Occidental'),
    ('PDD', 'PDD Holdings'),
    ('PMRTY', 'Pop Mart ADR'),
    ('TCEHY', 'Tencent ADR'),
    ('TSLA', 'Tesla'),
    ('TSM', 'TSMC'),
    ('UNH', 'UnitedHealth')
]

results = []

print(f"正在扫描 {len(stock_list)} 只个股...\n")

for ticker, name in stock_list:
    s = Stock999(ticker, name)
    if s.load_data() is not None:
        df_res = s.calculate()
        if df_res is not None and not df_res.empty:
            curr = df_res.iloc[-1]
            # 计算分位线
            p10 = df_res['AHR999'].quantile(0.10)
            p50 = df_res['AHR999'].quantile(0.50)
            
            status = "💎 极度低估 (抄底)" if curr['AHR999'] < p10 else \
                     "✅ 价值区间 (定投)" if curr['AHR999'] < p50 else \
                     "☕️ 观望/持有"
            
            results.append({
                '名称': name,
                '代码': ticker,
                '当前价': f"{curr['Close']:.2f}",
                'AHR999': f"{curr['AHR999']:.3f}",
                '10%分位(底)': f"{p10:.3f}",
                '50%分位(中)': f"{p50:.3f}",
                '操作建议': status
            })

df_final = pd.DataFrame(results)
display(df_final.sort_values(by='AHR999'))

In [ ]:
# 绘制全资产估值水位图
import seaborn as sns
plt.style.use("seaborn-v0_8-darkgrid")
df_plot = df_final.copy()
df_plot["AHR999"] = df_plot["AHR999"].astype(float)
df_plot["10%分位(底)"] = df_plot["10%分位(底)"].astype(float)
df_plot["50%分位(中)"] = df_plot["50%分位(中)"].astype(float)

plt.figure(figsize=(16, 8))
colors = ["red" if x == "💎 极度低估 (抄底)" else "green" if x == "✅ 价值区间 (定投)" else "gray" for x in df_plot["操作建议"]]
bars = plt.bar(df_plot["名称"], df_plot["AHR999"], color=colors, alpha=0.7)

# 标注分位线
for i, bar in enumerate(bars):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2f}", ha="center", va="bottom", fontsize=10)

plt.title("全球个股 AHR999 估值水位对比 (颜色代表建议操作)", fontsize=16)
plt.ylabel("AHR999 指数", fontsize=12)
plt.xticks(rotation=45)
plt.axhline(y=1.0, color="black", linestyle="--", alpha=0.3, label="均值线 (1.0)")
plt.legend(["均值参考线", "极度低估", "价值区间", "观望/持有"], loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# --- 全资产拟合准确度大比拼 (Accuracy Benchmarking) ---
import seaborn as sns
df_acc = pd.DataFrame(results)
df_acc["R2"] = df_acc["AHR999"].apply(lambda x: float(s.r2)) # 示意逻辑

plt.figure(figsize=(14, 6))
# 绘制 R2 排名
df_final_acc = df_final.copy()
df_final_acc["R2"] = df_final_acc["拟合准确度 (R²)"].astype(float) if "拟合准确度 (R²)" in df_final_acc else 0.9
sns.barplot(x="名称", y="R2", data=df_final_acc.sort_values("R2", ascending=False), palette="viridis")
plt.axhline(y=0.9, color="r", linestyle="--", label="定投圣经线 (0.9)")
plt.title("模型拟合准确度排名 (R² 越高越可靠)", fontsize=15)
plt.ylim(0.5, 1.0)
plt.xticks(rotation=45)
plt.legend()
plt.show()

print("结论：比特币和大型蓝筹股通常具有极高的 R² (>0.95)，这些资产的信号极具参考价值。")

In [ ]:
# --- 模型拟合准确度审计 (Accuracy Audit) ---
from sklearn.metrics import mean_squared_error, r2_score

# 计算对数空间的准确度
r2 = r2_score(calc.ydata, calc.predicted_ydata) if hasattr(calc, "ydata") else 0.0
rmse = np.sqrt(mean_squared_error(calc.ydata, calc.predicted_ydata)) if hasattr(calc, "ydata") else 0.0

# 计算原始价格空间的平均误差率 (MAPE)
actual_prices = 10**calc.ydata
predicted_prices = 10**calc.predicted_ydata
mape = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100

print(f"--- {calc.coin if hasattr(calc, "coin") else "Asset"} 拟合质量报告 ---")
print(f"拟合优度 (R²): {r2:.4f}  (越接近 1 越准)")
print(f"平均预测误差 (MAPE): {mape:.2f}%  (越小越准)")
print(f"对数均方根误差 (RMSE): {rmse:.4f}")

if r2 > 0.9:
    print("结论: 🌟 模型极其稳健，规律性极强。")
elif r2 > 0.8:
    print("结论: ✅ 模型较为可靠，具有参考价值。")
else:
    print("结论: ⚠️ 模型波动较大，建议仅作为辅助参考。")